# Tabular Training: Model Pool and Prediction Generation

Trains the fixed 15-model tabular pool on one dataset and writes the prediction
JSON consumed by the experiment pipeline. Dataset loading is specific to the
source (here OpenML "letter", i.e. the UCI Letter Recognition dataset, 26
classes); the pool, sizing rule, and split protocol follow the paper.

Sizing: N_test_min = ceil(20/p_min), N_test_target = 10 N_test_min,
N_train = min(ceil(500/p_min), N - N_test_target); stratified, no rebalancing,
10% validation for checkpoint selection. Output schema matches src/io_utils.py.


In [ ]:
# ============================================================
# TABULAR MODEL POOL v2 — 15 ARCHITECTURES (DEFINITION ONLY)
# Same-data / same-preprocess / same-loss regime
# Diversity comes from architecture + capacity/regularization spectrum only.
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.naive_bayes import GaussianNB

# ----------------------------
# Linear models (regularization path)
# ----------------------------
tab_logreg_c_0p001 = LogisticRegression(
    penalty="l2",
    C=0.001,
    solver="lbfgs",
    max_iter=5000,
    multi_class="auto",
    n_jobs=-1,
)

tab_logreg_c_0p01 = LogisticRegression(
    penalty="l2",
    C=0.01,
    solver="lbfgs",
    max_iter=5000,
    multi_class="auto",
    n_jobs=-1,
)

tab_logreg_c_0p1 = LogisticRegression(
    penalty="l2",
    C=0.1,
    solver="lbfgs",
    max_iter=5000,
    multi_class="auto",
    n_jobs=-1,
)

tab_logreg_c_1 = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=5000,
    multi_class="auto",
    n_jobs=-1,
)

tab_logreg_c_10 = LogisticRegression(
    penalty="l2",
    C=10.0,
    solver="lbfgs",
    max_iter=5000,
    multi_class="auto",
    n_jobs=-1,
)

tab_linear_svm_c_0p1 = LinearSVC(
    C=0.1,
    max_iter=10000,
)

# ----------------------------
# k-NN family (kept)
# ----------------------------
tab_knn_3  = KNeighborsClassifier(n_neighbors=3)
tab_knn_15 = KNeighborsClassifier(n_neighbors=15)
tab_knn_50 = KNeighborsClassifier(n_neighbors=50)

# ----------------------------
# Decision Trees (capacity spectrum)
# ----------------------------
tab_tree_depth_1   = DecisionTreeClassifier(max_depth=1,  random_state=42)
tab_tree_depth_3   = DecisionTreeClassifier(max_depth=3,  random_state=42)
tab_tree_depth_8   = DecisionTreeClassifier(max_depth=8,  random_state=42)
tab_tree_unpruned  = DecisionTreeClassifier(max_depth=None, random_state=42)

# ----------------------------
# Random Forests (weak vs strong)
# ----------------------------
tab_rf_20_depth_6 = RandomForestClassifier(
    n_estimators=20,
    max_depth=6,
    n_jobs=-1,
    random_state=42,
)

tab_rf_200 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)

# ----------------------------
# ExtraTrees (strong)
# ----------------------------
tab_extratrees_200 = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)

# ----------------------------
# Boosting (weak vs strong)
# ----------------------------
tab_gb_50_depth_2 = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=2,
    random_state=42,
)

tab_hgbt_300 = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=300,
    max_depth=None,
    l2_regularization=1e-3,
    early_stopping=False,
    random_state=42,
)

# ----------------------------
# Naive Bayes (kept as generative baseline)
# ----------------------------
tab_gnb = GaussianNB()

# ----------------------------
# Tabular MLP configs (unchanged)
# (Torch models defined later)
# ----------------------------
tab_mlp_small = {
    "type": "mlp",
    "hidden_dims": [64, 32],
    "dropout": 0.2,
}

tab_mlp_large = {
    "type": "mlp",
    "hidden_dims": [256, 128, 64],
    "dropout": 0.2,
}


In [ ]:
# ============================================================
# TABULAR DATASET — RATIONALITY CHECK ONLY
# Dataset: Letter Recognition (UCI / OpenML)
# Keeps your change: N_train_max = ceil(500 / p_min)
# ============================================================

import numpy as np
from sklearn.datasets import fetch_openml

# ----------------------------
# Load dataset
# ----------------------------
letter = fetch_openml(name="letter", version=1, as_frame=True)
df = letter.frame.copy()

# Robust target detection (OpenML usually uses "class" here)
if "class" in df.columns:
    target_col = "class"
elif "target" in df.columns:
    target_col = "target"
else:
    target_col = df.columns[-1]

y_raw = df[target_col].values

# Labels + counts
labels, counts = np.unique(y_raw, return_counts=True)
N_total = len(y_raw)
p_min = counts.min() / N_total

# ----------------------------
# Compute size constraints
# ----------------------------
N_test_min = int(np.ceil(20 / p_min))
N_test_target = 10 * N_test_min

N_train_min = int(np.ceil(50 / p_min))
N_train_max = int(np.ceil(500 / p_min))  # <-- your change (500)

N_train = min(N_train_max, N_total - N_test_target)

# ----------------------------
# Print summary
# ----------------------------
print("=== DATASET SUMMARY ===")
print("Dataset: Letter Recognition (OpenML: 'letter')")
print(f"Target column: {target_col}")
print(f"Total samples (N): {N_total}")
print(f"Num classes: {len(labels)}")
print(f"p_min: {p_min:.6f}")
print()

print("=== SIZE CONSTRAINTS ===")
print(f"N_test_min     = ceil(20 / p_min)  = {N_test_min}")
print(f"N_test_target  = 10 * N_test_min   = {N_test_target}")
print(f"N_train_min    = ceil(50 / p_min)  = {N_train_min}")
print(f"N_train_max    = ceil(500 / p_min) = {N_train_max}")
print(f"Chosen N_train = min(N_train_max, N - N_test_target) = {N_train}")
print()

# ----------------------------
# Rationality check
# ----------------------------
if N_train > N_train_min and (N_train + N_test_target) <= N_total:
    print("✅ DATASET PASSES nTrain / nTest RATIONALITY CHECK")
else:
    print("❌ DATASET FAILS nTrain / nTest RATIONALITY CHECK")


In [ ]:
# ============================================================
# TABULAR DATASET — SPLIT + PREPROCESS
# Dataset: Letter Recognition (UCI / OpenML: "letter")
# Uses your rules:
#   - Numerical: median impute + StandardScaler
#   - Categorical: most_frequent impute + OneHotEncoder(ignore)
# Split policy:
#   - exact N_TEST_TARGET test (stratified)
#   - exact N_TRAIN_TOTAL train+val from remaining pool (stratified)
#   - 10% val from train+val (stratified)
# Assumes already computed:
#   N_test_target, N_train
# Assumes df already loaded from fetch_openml("letter", ...)
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ---- Use your computed values ----
N_TEST_TARGET = int(N_test_target)
N_TRAIN_TOTAL = int(N_train)
VAL_FRAC = 0.10
RANDOM_STATE = 42

# ---- Letter Recognition target ----
# OpenML "letter" typically uses "class" as target (A..Z)
if "class" in df.columns:
    target_col = "class"
elif "target" in df.columns:
    target_col = "target"
else:
    target_col = df.columns[-1]

X = df.drop(columns=[target_col])
y_raw = df[target_col].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
num_classes = len(le.classes_)

# ---- Exact split: choose N_TEST_TARGET stratified ----
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=N_TEST_TARGET, stratify=y, random_state=RANDOM_STATE
)

# ---- From remaining pool choose exactly N_TRAIN_TOTAL stratified ----
X_trainval, _, y_trainval, _ = train_test_split(
    X_pool, y_pool, train_size=N_TRAIN_TOTAL, stratify=y_pool, random_state=RANDOM_STATE
)

# ---- Split trainval into train/val (10% val) stratified ----
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_FRAC, stratify=y_trainval, random_state=RANDOM_STATE
)

# ---- Build preprocessing (your rules) ----
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

if len(cat_cols) > 0:
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols),
        ],
        remainder="drop",
    )
else:
    preprocess = ColumnTransformer(
        transformers=[("num", numeric_transformer, num_cols)],
        remainder="drop",
    )

# ---- Fit on TRAIN only; transform all ----
X_train_proc = preprocess.fit_transform(X_train)
X_val_proc   = preprocess.transform(X_val)
X_test_proc  = preprocess.transform(X_test)

# ---- Quick reporting ----
def bincount_dict(arr, k):
    bc = np.bincount(arr, minlength=k)
    return {int(i): int(bc[i]) for i in range(k)}

print("=== SPLITS ===")
print(f"Dataset: Letter Recognition (OpenML 'letter')")
print(f"Target column: {target_col}")
print(f"Train+Val: {len(y_trainval)} (target {N_TRAIN_TOTAL})")
print(f"  Train:   {len(y_train)}")
print(f"  Val:     {len(y_val)} (10% of train+val)")
print(f"Test:      {len(y_test)} (target {N_TEST_TARGET})")
print("Class counts:")
print("  Train:", bincount_dict(y_train, num_classes))
print("  Val:  ", bincount_dict(y_val,   num_classes))
print("  Test: ", bincount_dict(y_test,  num_classes))
print()

print("=== PREPROCESS ===")
print("Numerical: median impute + StandardScaler")
if len(cat_cols) > 0:
    ohe = preprocess.named_transformers_["cat"].named_steps["onehot"]
    num_ohe_features = int(sum(len(cats) for cats in ohe.categories_))
    print("Categorical: most_frequent impute + OneHotEncoder(handle_unknown='ignore')")
    print(f"Num cols: {len(num_cols)} | Cat cols: {len(cat_cols)}")
    print(f"One-hot features: {num_ohe_features} | Total features: {X_train_proc.shape[1]}")
else:
    print("Categorical: none detected (cat_cols is empty) -> no one-hot applied")
    print(f"Num cols: {len(num_cols)} | Cat cols: 0")
    print(f"Total features: {X_train_proc.shape[1]}")

print("Matrices:")
print(f"  X_train_proc: {X_train_proc.shape}")
print(f"  X_val_proc:   {X_val_proc.shape}")
print(f"  X_test_proc:  {X_test_proc.shape}")


In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.utils.class_weight import compute_sample_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# TRAINING CELL (Tabular Pool v2)
# Assumes already defined in earlier cells:
#   - split/preprocess cell already ran, producing:
#       X_train_proc, X_val_proc, y_train, y_val, num_classes
#   - model objects exist from Cell 1 for the names used below
# ============================================================

# ----------------------------
# Utilities: densify with feasibility check
# ----------------------------
def dense_bytes_estimate(X_sparse):
    return int(X_sparse.shape[0]) * int(X_sparse.shape[1]) * 4  # float32 bytes

def to_dense_safe(X, max_gb: float = 2.0):
    """
    Convert sparse -> dense if estimated size <= max_gb.
    Returns (X_dense, ok_bool).
    """
    if not sp.issparse(X):
        return (X.astype(np.float32) if hasattr(X, "astype") else X), True
    est_gb = dense_bytes_estimate(X) / (1024**3)
    if est_gb > max_gb:
        return None, False
    return X.toarray().astype(np.float32), True

# ----------------------------
# Class weights / sample weights (TRAIN ONLY)
# ----------------------------
sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)

num_classes = int(len(np.unique(y_train)))
class_counts = np.bincount(y_train, minlength=num_classes).astype(np.float64)
class_weights = (class_counts.sum() / (num_classes * np.maximum(class_counts, 1.0)))
class_weights_t = torch.tensor(class_weights, dtype=torch.float32)

print("=== CLASS IMBALANCE HANDLING (TRAIN ONLY) ===")
print(f"Class counts (train): {class_counts.tolist()}")
print(f"Class weights (MLP):  {class_weights.tolist()}")
print("Sklearn: class_weight='balanced' where supported; otherwise sample_weight if supported.")
print("kNN: unweighted.")
print()

# ----------------------------
# Gather model objects (NO re-definition)
# NOTE: This dict defines the EXACT sklearn pool used in training.
# Total models per run = len(models) + len(mlp_configs) = 13 + 2 = 15
# ----------------------------
KNN_NAMES = {"tab_knn_15"}

# Models here support class_weight='balanced' via set_params, or we can use sample_weight.
CLASS_WEIGHT_COMPATIBLE = {
    "tab_logreg_c_0p001",
    "tab_logreg_c_0p01",
    "tab_logreg_c_0p1",
    "tab_logreg_c_1",
    "tab_linear_svm_c_0p1",
    "tab_tree_depth_1",
    "tab_tree_depth_3",
    "tab_tree_unpruned",
    "tab_rf_20_depth_6",
    "tab_rf_200",
}

# These models prefer/require dense inputs when preprocessing produces sparse matrices
PREFER_DENSE = {
    "tab_hgbt_300",
    "tab_knn_15",
    "tab_tree_depth_1",
    "tab_tree_depth_3",
    "tab_tree_unpruned",
    "tab_rf_20_depth_6",
    "tab_rf_200",
    "tab_gnb",
}

models = {
    # Linear (regularization spectrum)
    "tab_logreg_c_0p001": tab_logreg_c_0p001,
    "tab_logreg_c_0p01":  tab_logreg_c_0p01,
    "tab_logreg_c_0p1":   tab_logreg_c_0p1,
    "tab_logreg_c_1":     tab_logreg_c_1,
    "tab_linear_svm_c_0p1": tab_linear_svm_c_0p1,

    # kNN / generative
    "tab_knn_15": tab_knn_15,
    "tab_gnb": tab_gnb,

    # Trees
    "tab_tree_depth_1": tab_tree_depth_1,
    "tab_tree_depth_3": tab_tree_depth_3,
    "tab_tree_unpruned": tab_tree_unpruned,

    # Forests + strong booster
    "tab_rf_20_depth_6": tab_rf_20_depth_6,
    "tab_rf_200": tab_rf_200,
    "tab_hgbt_300": tab_hgbt_300,
}

mlp_configs = {
    "tab_mlp_small": tab_mlp_small,
    "tab_mlp_large": tab_mlp_large,
}

# ----------------------------
# Ensure class_weight="balanced" where supported (and not kNN)
# ----------------------------
for name, model in models.items():
    if name in KNN_NAMES:
        continue
    if name in CLASS_WEIGHT_COMPATIBLE:
        try:
            if "class_weight" in model.get_params():
                model.set_params(class_weight="balanced")
        except Exception:
            pass

# ----------------------------
# Train classical models (TRAIN ONLY)
# - Try sample_weight first; if TypeError -> fit without
# - Uses sparse when possible; falls back to dense if sparse fails
# - Skips only if densification infeasible (>2GB)
# ----------------------------
trained_models = {}
skipped_models = {}

Xtr_sparse = X_train_proc
Ytr = y_train

for name, model in models.items():
    print(f"[TRAIN] {name}")

    use_dense = (name in PREFER_DENSE) and sp.issparse(Xtr_sparse)
    if use_dense:
        Xtr, ok = to_dense_safe(Xtr_sparse, max_gb=2.0)
        if not ok:
            skipped_models[name] = "Skipped: densification estimated >2GB"
            print("  -> SKIP (densification infeasible)")
            continue
    else:
        Xtr = Xtr_sparse

    # kNN: no weighting
    if name in KNN_NAMES:
        try:
            model.fit(Xtr, Ytr)
            trained_models[name] = model
            print("  -> trained (unweighted)")
        except Exception as e:
            skipped_models[name] = f"Fit failed: {repr(e)}"
            print(f"  -> SKIP (fit failed): {repr(e)}")
        continue

    # Try sample_weight when supported
    try:
        model.fit(Xtr, Ytr, sample_weight=sample_weight_train)
        trained_models[name] = model
        print("  -> trained (sample_weight)")
    except TypeError:
        try:
            model.fit(Xtr, Ytr)
            trained_models[name] = model
            print("  -> trained (no sample_weight supported)")
        except Exception as e:
            skipped_models[name] = f"Fit failed: {repr(e)}"
            print(f"  -> SKIP (fit failed): {repr(e)}")
    except Exception as e:
        # Fallback: if sparse failed, try dense
        if (not use_dense) and sp.issparse(Xtr_sparse):
            Xtr2, ok = to_dense_safe(Xtr_sparse, max_gb=2.0)
            if not ok:
                skipped_models[name] = f"Sparse fit failed; densification infeasible: {repr(e)}"
                print("  -> SKIP (sparse failed; densify infeasible)")
                continue
            try:
                model.fit(Xtr2, Ytr, sample_weight=sample_weight_train)
                trained_models[name] = model
                print("  -> trained (after sparse->dense fallback, sample_weight)")
            except TypeError:
                try:
                    model.fit(Xtr2, Ytr)
                    trained_models[name] = model
                    print("  -> trained (after sparse->dense fallback, no sample_weight)")
                except Exception as e2:
                    skipped_models[name] = f"Fit failed (dense fallback): {repr(e2)}"
                    print(f"  -> SKIP (dense fallback failed): {repr(e2)}")
            except Exception as e2:
                skipped_models[name] = f"Fit failed (dense fallback): {repr(e2)}"
                print(f"  -> SKIP (dense fallback failed): {repr(e2)}")
        else:
            skipped_models[name] = f"Fit failed: {repr(e)}"
            print(f"  -> SKIP (fit failed): {repr(e)}")

print("\n=== CLASSICAL TRAINING SUMMARY ===")
print(f"Trained classical models: {len(trained_models)}/{len(models)}")
if skipped_models:
    print("Skipped:")
    for k, v in skipped_models.items():
        print(f"  {k}: {v}")

# ----------------------------
# Train Torch MLPs (TRAIN ONLY)
# - class-weighted CrossEntropyLoss
# - early stopping on VAL macro-F1 (patience=10), restore best checkpoint
# - Uses DENSE features (MLP requires dense)
# ----------------------------
from sklearn.metrics import f1_score  # used only for early stopping signal (not reporting)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_EPOCHS = 50
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10

Xtr_mlp, ok1 = to_dense_safe(X_train_proc, max_gb=2.0)
Xva_mlp, ok2 = to_dense_safe(X_val_proc, max_gb=2.0)
if not (ok1 and ok2):
    raise RuntimeError("MLP training requires dense features but densification was infeasible under the 2GB cap.")

class NPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

class TabMLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

trained_mlps = {}

for mlp_name, cfg in mlp_configs.items():
    print(f"\n[TRAIN] {mlp_name} (Torch MLP)")

    in_dim = Xtr_mlp.shape[1]
    model = TabMLP(in_dim, cfg["hidden_dims"], num_classes, cfg["dropout"]).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss(weight=class_weights_t.to(DEVICE))

    tr_loader = DataLoader(NPDataset(Xtr_mlp, y_train), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(NPDataset(Xva_mlp, y_val), batch_size=BATCH_SIZE, shuffle=False)

    best_f1 = -1.0
    best_state = None
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        val_preds = []
        val_true = []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(DEVICE)
                logits = model(xb).cpu().numpy()
                val_preds.append(logits.argmax(axis=1))
                val_true.append(yb.numpy())
        val_preds = np.concatenate(val_preds)
        val_true = np.concatenate(val_true)
        val_f1 = float(f1_score(val_true, val_preds, average="macro"))

        print(f"  epoch {epoch:02d} | val_macro_f1={val_f1:.4f} | best={best_f1:.4f} | no_improve={no_improve}")

        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print("  -> early stopping")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    trained_mlps[mlp_name] = model
    print("  -> trained (best checkpoint restored)")

print("\n=== MLP TRAINING SUMMARY ===")
print(f"Trained MLPs: {len(trained_mlps)}/2")
print("\nDONE. (No test prediction / no evaluation in this cell.)")


In [ ]:
# ============================================================
# EVAL + PRINT PERFORMANCE + SAVE JSON TO DRIVE (LETTER)
# Assumes already exist:
#   trained_models (sklearn) , trained_mlps (torch)
#   X_test_proc, y_test, num_classes
# ============================================================

import os, json
import numpy as np
import scipy.sparse as sp
import torch
from sklearn.metrics import accuracy_score, f1_score

# ----------------------------
# Paths
# ----------------------------

DATASET_NAME = "uci_letter"   # <-- CHANGED
BASE_DIR = "data/predictions"
OUT_DIR = os.path.join(BASE_DIR, DATASET_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_JSON = os.path.join(OUT_DIR, f"{DATASET_NAME}_test_predictions_logits.json")

# ----------------------------
# Helpers
# ----------------------------
def to_dense(X):
    return X.toarray() if sp.issparse(X) else np.asarray(X)

def _fix_binary_logits(s):
    # if decision_function returns shape (N,), convert to (N,2)
    s = np.asarray(s)
    if s.ndim == 1:
        s = s.reshape(-1, 1)
        s = np.hstack([-s, s])
    return s

def get_logits_sklearn(model, X, num_classes):
    # Prefer decision_function when available
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        s = np.asarray(s)
        # binary special-case
        if num_classes == 2:
            return _fix_binary_logits(s)
        # multiclass: should already be (N, C)
        if s.ndim == 1:
            raise RuntimeError("Unexpected 1D decision_function output for multiclass.")
        return s

    # Next: log-probabilities if available
    if hasattr(model, "predict_log_proba"):
        lp = np.asarray(model.predict_log_proba(X))
        if lp.shape[1] != num_classes:
            raise RuntimeError(f"predict_log_proba returned {lp.shape[1]} cols, expected {num_classes}.")
        return lp

    # Next: probabilities -> take log for 'logits-like' comparable scale
    if hasattr(model, "predict_proba"):
        p = np.asarray(model.predict_proba(X))
        if p.shape[1] != num_classes:
            raise RuntimeError(f"predict_proba returned {p.shape[1]} cols, expected {num_classes}.")
        return np.log(p + 1e-12)

    raise RuntimeError("Model has no decision_function / predict_*proba to extract logits.")

DENSE_MODELS = {
    "tab_hgbt_300",
    "tab_knn_15",
    "tab_tree_depth_1", "tab_tree_depth_3", "tab_tree_unpruned",
    "tab_rf_20_depth_6", "tab_rf_200",
    "tab_gnb",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Evaluate + collect
# ----------------------------
results = {
    "dataset": DATASET_NAME,
    "num_classes": int(num_classes),
    "y_test": y_test.tolist(),   # ✅ true labels included
    "models": {}
}

print("=== TEST PERFORMANCE (accuracy, macro_f1) ===")

# Classical (sklearn)
for name, model in trained_models.items():
    Xte = to_dense(X_test_proc) if (name in DENSE_MODELS) else X_test_proc
    logits = get_logits_sklearn(model, Xte, num_classes)

    logits = np.asarray(logits)
    if logits.ndim == 1:
        raise RuntimeError(f"{name}: logits came out 1D; expected (N, C).")

    pred = logits.argmax(axis=1)

    acc = float(accuracy_score(y_test, pred))
    mf1 = float(f1_score(y_test, pred, average="macro"))

    print(f"{name:22s}  acc={acc:.4f}  macro_f1={mf1:.4f}")

    results["models"][name] = {
        "logits": logits.tolist(),
        "y_pred": pred.tolist(),
        "metrics": {"accuracy": acc, "macro_f1": mf1},
    }

# Torch MLPs
for name, model in trained_mlps.items():
    Xte = to_dense(X_test_proc).astype(np.float32)
    model.eval()
    logits_chunks = []

    with torch.no_grad():
        for i in range(0, Xte.shape[0], 256):
            xb = torch.tensor(Xte[i:i+256], dtype=torch.float32).to(DEVICE)
            logits_chunks.append(model(xb).cpu().numpy())

    logits = np.vstack(logits_chunks)
    pred = logits.argmax(axis=1)

    acc = float(accuracy_score(y_test, pred))
    mf1 = float(f1_score(y_test, pred, average="macro"))

    print(f"{name:22s}  acc={acc:.4f}  macro_f1={mf1:.4f}")

    results["models"][name] = {
        "logits": logits.tolist(),
        "y_pred": pred.tolist(),
        "metrics": {"accuracy": acc, "macro_f1": mf1},
    }

print("\nModels evaluated:", len(results["models"]))
print("Saving to:", OUT_JSON)

with open(OUT_JSON, "w") as f:
    json.dump(results, f)

print("✅ Saved.")
print("✅ True labels are included in JSON under key: 'y_test'")


Letter Recognition exhibits a distinct form of expert structure driven by its large number of classes. While several models achieve high overall performance, the pairwise disagreement matrix reveals systematic differences between tree-based ensembles, distance-based methods, and neural models. Linear classifiers and shallow trees perform substantially worse and disagree broadly with stronger models, indicating biased or oversimplified decision boundaries.

This structured heterogeneity suggests the presence of multiple complementary experts rather than a single dominant model family. Consequently, restricting aggregation to a selected expert subset is expected to outperform majority voting over the full model pool, particularly in terms of macro-F1.